# CS781 Phase 5 · Batching and memory benchmark

This experiment compares the original document-at-a-time encoder with a version that batches valid chunks from several cases. Each chunk remains independent in the Transformer. Its representation is returned to the correct case before the existing BiGRU and label-wise attention head.

The main comparison is the prefix model with LoRMA+. A short encoder–decoder comparison is included because its original timing was also relatively slow.

All prefix configurations process **64 cases per optimizer update**. Chunk length, document coverage, task head and adaptation parameters are held fixed. The timing probes use zero-learning-rate AdamW updates and discard the models afterwards.

**Inputs**

- GPU T4 ×2.
- The completed preparation at `/kaggle/working/cs781_phase5`, or the exported `phase5_lsi_prepared_data.zip` attached as a dataset. Expanded Kaggle datasets are supported.
- The original Phase 1 prefix checkpoint. The default encoder–decoder comparison also needs its original Phase 1 checkpoint.

The notebook contains its source modules. No additional code package, tokenizer upload, Hugging Face token or data download is required when the prepared cache is available. It does not repeat tokenization.


### Kaggle inputs

Attach the packed data from [preparation](phase5_preparation.ipynb) and the Phase 1 checkpoints for the architectures being benchmarked. Set `PREPARED_INPUT` and `CHECKPOINT_INPUTS` if several inputs are mounted.


## 1. Experiment settings

The physical case batch is shared across both GPUs. The chunk microbatch is the maximum number of chunks encoded together **on each GPU**.

| Physical cases | Accumulation steps | Cases per optimizer update |
|---:|---:|---:|
| 4 | 16 | 64 |
| 8 | 8 | 64 |
| 16 | 4 | 64 |
| 32 | 2 | 64 |

Five hours is the training target. The results distinguish five hours of optimizer work from a five-hour training loop that also reserves time for development evaluation and checkpoint operations. Setup, teacher-cache generation and final test export are separate.

Automatic discovery first reuses staged checkpoints and the existing packed cache. Exact overrides are needed only when an input is ambiguous or stored elsewhere. Set `QUICK_OTHER_ARCHITECTURES = []` for the prefix benchmark alone.


In [1]:
from pathlib import Path

PROJECT_ROOT = Path("/kaggle/working/cs781_phase5")
BENCHMARK_ROOT = Path("/kaggle/working/phase5_batching_benchmark")
INPUT_ROOT = Path("/kaggle/input")

PREPARED_INPUT = None
CHECKPOINT_INPUTS = {
    "prefix": None,
    "encoder_decoder": None,
    "decoder": None,
    "efficient": None,
}

QUICK_OTHER_ARCHITECTURES = ["encoder_decoder"]
# Optional additional quick comparisons: "decoder" and "efficient".

EFFECTIVE_CASES = 64
TARGET_TRAINING_HOURS = 5.0
MAX_RUN_HOURS = 5.0
MEMORY_LIMIT_GIB = 12.0

SWEEP_UPDATES = 2
CONFIRM_UPDATES = 5
FULL_DEV_EVALUATIONS = 5
SAVE_AND_METRIC_RESERVE_MINUTES = 10
FORCE_RERUN = False

GEOMETRY = {"total_tokens": 512, "overlap": 64, "max_chunks": 4, "policy": "uniform"}
MODEL_SETTINGS = {'seed': 42,
 'head_seed': 42,
 'gradient_checkpointing': True,
 'chunk_microbatch': 2,
 'backbone_dropout': 0.1,
 'head_dropout': 0.3,
 'decoder_upper_layers': 2,
 'adapter_bottleneck': 64,
 'adapter_dropout': 0.05,
 'adapter_train_layer_norm': True,
 'lorma_rank': 8,
 'lorma_alpha': 8,
 'lorma_dropout': 0.05,
 'student_layers': [0, 7, 15],
 'projection_dim': 256,
 'context_hidden': 128,
 'attention_hidden': 64}


## 2. Source snapshot

The bundled source expands into readable Python files under the benchmark directory. `lsi_batched_model.py` contains the batching change. The original model implementation is retained for the baseline, and original preparation reports remain intact.


In [2]:
import base64
import hashlib
import io
import zipfile

SOURCE_ARCHIVE_B64 = "..."
source_bytes = base64.b64decode(SOURCE_ARCHIVE_B64)
assert hashlib.sha256(source_bytes).hexdigest() == "6278950d5ba9b2533987ee68a7a4e39b279e9b179a5fbbdc39b85ea6792ed6c5"
BENCHMARK_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(io.BytesIO(source_bytes)) as archive:
    for member in archive.infolist():
        destination = (BENCHMARK_ROOT / member.filename).resolve()
        if not destination.is_relative_to(BENCHMARK_ROOT.resolve()):
            raise ValueError("Unexpected source archive path")
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.read(member))

print("Readable source files", BENCHMARK_ROOT / "src")


Readable source files /kaggle/working/phase5_batching_benchmark/src


## 3. Benchmark configuration

The prefix sweep compares the original batch with cross-case chunk batches from 4 to 64. It tests physical case batches up to 32 and two configurations without gradient checkpointing. The smaller no-checkpoint case batch is included in case its activation memory makes a larger one impractical.

Correctness checks compare FP32 logits, chunk attention and every trainable gradient, including a two-GPU comparison. Dropout is disabled for these comparisons because regrouping changes the order of random draws. Training timings use the original dropout settings.

Each timing process performs a maximum-shape forward/backward check using four 512-position chunks per case. It then runs one untimed warm-up and the measured accumulated updates. All configurations use the same sampled case groups. The two fastest viable prefix configurations are measured again on a separate sample.

The memory allowance applies to **each GPU**, using peak reserved memory. An out-of-memory candidate is recorded; the following candidate starts in a clean process. No training checkpoint is written.


In [3]:
import json
import subprocess
import sys

settings = {
    "project_root": str(PROJECT_ROOT),
    "input_root": str(INPUT_ROOT),
    "output_root": str(BENCHMARK_ROOT / "results"),
    "prepared_input": PREPARED_INPUT,
    "checkpoint_inputs": CHECKPOINT_INPUTS,
    "quick_other_architectures": QUICK_OTHER_ARCHITECTURES,
    "deduplicate_training": True,
    "seed": 42,
    "amp": True,
    "effective_cases": EFFECTIVE_CASES,
    "geometry": GEOMETRY,
    "model_settings": MODEL_SETTINGS,
    "sweep_updates": SWEEP_UPDATES,
    "confirm_updates": CONFIRM_UPDATES,
    "sweep_inference_cases": 64,
    "confirm_inference_cases": 256,
    "memory_limit_gib": MEMORY_LIMIT_GIB,
    "worker_timeout_seconds": 900,
    "target_training_hours": TARGET_TRAINING_HOURS,
    "max_run_hours": MAX_RUN_HOURS,
    "full_dev_evaluations": FULL_DEV_EVALUATIONS,
    "save_and_metric_reserve_minutes": SAVE_AND_METRIC_RESERVE_MINUTES,
    "force_rerun": FORCE_RERUN,
}
config_path = BENCHMARK_ROOT / "benchmark_settings.json"
config_path.write_text(json.dumps(settings, indent=2), encoding="utf-8")

try:
    inventory = subprocess.run(
        ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free", "--format=csv"],
        capture_output=True, text=True, check=True,
    )
    print(inventory.stdout)
except (FileNotFoundError, subprocess.CalledProcessError):
    print("GPU inventory unavailable. The benchmark requires two CUDA GPUs.")

print("Effective cases per update", EFFECTIVE_CASES)
print("Training target in hours", TARGET_TRAINING_HOURS)
print("Additional quick comparisons", QUICK_OTHER_ARCHITECTURES)


index, name, memory.total [MiB], memory.free [MiB]
0, Tesla T4, 15360 MiB, 14912 MiB
1, Tesla T4, 15360 MiB, 14912 MiB

Effective cases per update 64
Training target in hours 5.0
Additional quick comparisons ['encoder_decoder']


## 4. Run comparisons

Logs appear below after loading and each update. Candidate reports are saved immediately. Rerunning this cell reuses successful reports only when the complete input and source configuration matches. The worker timeout is per candidate.

The timings establish throughput and memory use. They do not estimate development F1 or prove that five hours of training will be sufficient for convergence.


In [4]:
import os
import signal

command = [
    sys.executable, "-u", str(BENCHMARK_ROOT / "src/lsi_batching_benchmark.py"),
    "--suite", str(config_path),
]
process = subprocess.Popen(
    command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, start_new_session=True,
)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    returncode = process.wait()
except BaseException:
    os.killpg(process.pid, signal.SIGTERM)
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        os.killpg(process.pid, signal.SIGKILL)
        process.wait()
    raise

if returncode != 0:
    raise RuntimeError("Benchmark stopped. Inspect the last log and saved reports before rerunning.")


Verifying original checkpoint: prefix
Verifying original checkpoint: encoder_decoder

Benchmark: prefix correctness
Loading prefix from the original Phase 1 checkpoint

Benchmark: prefix original_b8_c2_gc
Loading prefix from the original Phase 1 checkpoint
Maximum-shape forward/backward and optimizer check passed
Warm-up 29.410 seconds, loss 0.45719
Measured update 1/2 26.234 seconds, loss 0.44876
Measured update 2/2 26.386 seconds, loss 0.38594

Benchmark: prefix flat_b8_c4_gc
Loading prefix from the original Phase 1 checkpoint
Maximum-shape forward/backward and optimizer check passed
Warm-up 16.332 seconds, loss 0.45814
Measured update 1/2 14.712 seconds, loss 0.44920
Measured update 2/2 14.732 seconds, loss 0.38872

Benchmark: prefix flat_b8_c8_gc
Loading prefix from the original Phase 1 checkpoint
Maximum-shape forward/backward and optimizer check passed
Warm-up 11.683 seconds, loss 0.45667
Measured update 1/2 10.896 seconds, loss 0.44779
Measured update 2/2 11.104 seconds, loss 0.

## 5. Results and five-hour projections

`seconds_per_update` compares the cost of the same effective batch. The fastest confirmed prefix configuration determines its proposal. The secondary architecture compares the original, a moderate batched configuration and the prefix winner; it is not a complete architecture-specific sweep.

The conservative loop proposal reserves five full development passes plus ten minutes for metrics and checkpoint operations. These are explicit allowances, not measured checkpoint times. The LR horizon should be fixed from the chosen optimizer-update budget at launch and retained on resume.


In [5]:
import pandas as pd
from IPython.display import display

result_root = Path(settings["output_root"])
comparison_path = result_root / "comparison.csv"
if comparison_path.exists():
    comparisons = pd.read_csv(comparison_path)
    with pd.option_context("display.max_columns", None, "display.float_format", "{:.3f}".format):
        display(comparisons)
else:
    print("No timing table was produced. Inspect the correctness or input log.")

proposal_path = result_root / "recommended_training_settings.json"
proposals = json.loads(proposal_path.read_text()) if proposal_path.exists() else {}
budget_rows = []
for architecture, proposal in proposals.items():
    budget_rows.append({
        "architecture": architecture,
        "physical_cases": proposal["train_batch_cases"],
        "accumulation": proposal["accumulation_steps"],
        "chunks_per_gpu_call": proposal["chunk_microbatch"],
        "checkpointing": proposal["gradient_checkpointing"],
        "seconds_per_update": proposal["seconds_per_update"],
        "updates_in_5h_optimizer_work": proposal["five_hours_optimizer_work"]["updates"],
        "epochs_in_5h_optimizer_work": proposal["five_hours_optimizer_work"]["equivalent_epochs"],
        "updates_in_5h_loop": proposal["five_hour_loop_proposal"]["fixed_optimizer_updates"],
        "epochs_in_5h_loop": proposal["five_hour_loop_proposal"]["equivalent_epochs"],
        "teacher_train_cache_minutes": proposal["teacher_train_cache_estimated_minutes"],
    })
if budget_rows:
    with pd.option_context("display.max_columns", None, "display.float_format", "{:.3f}".format):
        display(pd.DataFrame(budget_rows))
print("Full configuration proposals", proposal_path)


,architecture,phase,configuration,status,physical_batch,accumulation,effective_cases,chunks_per_gpu_call,checkpointing,seconds_per_update,epoch_minutes,dev_minutes,peak_allocated_gib,peak_reserved_gib,memory_ok,error
0,prefix,sweep,original_b8_c2_gc,passed,8,8,64,2,True,26.310,292.916,14.141,2.791,3.082,True,NaN
1,prefix,sweep,flat_b8_c4_gc,passed,8,8,64,4,True,14.722,163.905,9.331,2.790,3.107,True,NaN
2,prefix,sweep,flat_b8_c8_gc,passed,8,8,64,8,True,11.000,122.470,8.364,2.793,3.039,True,NaN
3,prefix,sweep,flat_b8_c16_gc,passed,8,8,64,16,True,9.571,106.558,7.838,2.792,3.221,True,NaN
4,prefix,sweep,flat_b16_c16_gc,passed,16,4,64,16,True,8.524,94.897,7.566,2.793,3.262,True,NaN
5,prefix,sweep,flat_b16_c32_gc,passed,16,4,64,32,True,8.283,92.218,7.097,3.209,4.174,True,NaN
6,prefix,sweep,flat_b32_c32_gc,passed,32,2,64,32,True,7.502,83.526,6.550,4.269,4.385,True,NaN
7,prefix,sweep,flat_b32_c64_gc,passed,32,2,64,64,True,7.444,82.872,6.270,5.537,6.283,True,NaN
8,prefix,sweep,flat_b8_c8_no_gc,passed,8,8,64,8,False,7.639,85.047,8.394,9.840,10.268,True,NaN
9,prefix,sweep,flat_b4_c8_no_gc,passed,4,16,64,8,False,8.829,98.298,10.534,5.529,5.701,True,NaN


,architecture,physical_cases,accumulation,chunks_per_gpu_call,checkpointing,seconds_per_update,updates_in_5h_optimizer_work,epochs_in_5h_optimizer_work,updates_in_5h_loop,epochs_in_5h_loop,teacher_train_cache_minutes
0,prefix,32,2,64,True,7.653,2352,3.521,2045,3.062,24.025
1,encoder_decoder,32,2,64,True,3.466,5192,7.773,4769,7.140,12.242


Full configuration proposals /kaggle/working/phase5_batching_benchmark/results/recommended_training_settings.json


## 6. Export benchmark record

The diagnostics contain source snapshots, exact input identities, correctness checks, per-candidate logs, timing tables and configuration proposals. They exclude the packed dataset and model weights.

The reusable `phase5_lsi_prepared_data.zip` from preparation remains the data input for training.


In [6]:
from IPython.display import FileLink

diagnostics = PROJECT_ROOT.parent / "phase5_batching_benchmark_diagnostics.zip"
if diagnostics.is_file():
    print("Diagnostics path", diagnostics)
    print(f"Size {diagnostics.stat().st_size / 1024**2:.2f} MB")
    display(FileLink(str(diagnostics)))
else:
    print("No diagnostics archive yet. Check the benchmark log.")


Diagnostics path /kaggle/working/phase5_batching_benchmark_diagnostics.zip
Size 0.13 MB


/kaggle/working/phase5_batching_benchmark_diagnostics.zip

## Implementation record

The baseline is the checked Phase 5 preparation implementation. The new batching path only changes how independent chunks are grouped into Transformer calls. Original Phase 1 weights, tokenizer identity, LoRMA+ factors, adapters, readout rules, chunk geometry and the multi-label head are retained.

The source and checkpoint hashes are recorded with every request. CPU fixture checks cover all four architecture paths; the Kaggle run supplies the actual two-GPU equivalence, memory and timing measurements.
